# BasuraBin: comparing three pretrained CNNs for household waste segregation

**Computer Vision Comparison Project, West Visayas State University**

**Task:** image classification into the three household bins required by RA 9003 (Ecological Solid Waste Management Act):
**Biodegradable** (*nabubulok*), **Recyclable**, and **Non-biodegradable / Residual** (*di-nabubulok*).

**Research question:** *For a real-time camera app that sorts household waste into the three RA 9003 bins, which ImageNet-pretrained
architecture gives the best trade-off between accuracy, speed and size: a CNN built for phones (MobileNetV3), a compound-scaled CNN
(EfficientNet-B0) or a classic residual CNN (ResNet-18)?*

| Model | Year | Design idea | Size |
|---|---|---|---|
| A. MobileNetV3-Large | 2019 | built for phones: depthwise-separable convolutions, squeeze-and-excitation, h-swish | ~4 M params, 0.22 G MACs |
| B. EfficientNet-B0 | 2019 | compound scaling of depth, width and resolution; MBConv blocks | ~4 M params, 0.39 G MACs |
| C. ResNet-18 | 2016 | classic residual network with full 3×3 convolutions | ~11 M params, 1.8 G MACs |

All three load ImageNet-1k weights from torchvision and are fine-tuned with **the same images, the same train/val/test split
(`splits/*.csv`), the same preprocessing and the same training recipe** (`finetune.py`); only the architecture differs.

Run every cell from top to bottom (works in Google Colab and on a local Jupyter with a GPU).

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/nylla8444/comvis-final.git"
if IN_COLAB:
    if not os.path.exists("/content/comvis-final"):
        !git clone -q {REPO_URL} /content/comvis-final
    os.chdir("/content/comvis-final")
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

QUICK = False  # True = tiny epoch counts, only for checking that the notebook runs

import json, numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from IPython.display import Image as ShowImage, Markdown, display
from common import CLASSES, METHOD_INFO, METHOD_KEYS, MODELS_DIR, RESULTS_DIR, SPLITS_DIR, load_json, load_rgb, load_split, plot_history, set_seed
set_seed(42)
print("working dir:", os.getcwd())
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 1. Data

Two public datasets are merged and relabelled into the three bins (no Kaggle login needed):

| Source | Images | Link | Licence |
|---|---|---|---|
| Garbage Classification (12 classes), web photos | 15,515 | https://www.kaggle.com/datasets/mostafaabla/garbage-classification | see Kaggle page |
| RealWaste, real landfill photos (Wollongong, AU) | 4,752 | https://archive.ics.uci.edu/dataset/908/realwaste (mirror: kaggle.com/datasets/joebeachcapital/realwaste) | CC BY 4.0 |

Mapping to bins: biological, food organics, vegetation → **biodegradable**; paper, cardboard, metal, plastic, glass → **recyclable**;
trash, misc. trash, textile trash, clothes, shoes → **residual**. Batteries are dropped (special/hazardous waste under RA 9003).

`prepare_data.py` removes duplicate images (difference hash), caps each bin at 2,500 images sampled evenly across source classes
(so the 5,325 clothes photos cannot dominate the residual bin), splits 70/15/15 stratified by source class (seed 42), and stores
every image as a 256×256 centre crop.

**Our own photos (optional).** Webcam/phone photos saved with `python live.py` (keys 1/2/3) go to `data/raw/local/<bin>/`
(an extra test set, never trained on) or, with `--save-to train`, to `data/raw/local_train/<bin>/` (added to the training split
for all three models). Different objects are used in the two folders, so the local test measures new items.

In [ ]:
import download_data, prepare_data
download_data.main([])
prepare_data.main([])

In [ ]:
summary = load_json(SPLITS_DIR / "summary.json")
display(pd.DataFrame(summary["counts"]).T.assign(total=lambda d: d.sum(1)))
print("total images:", summary["total_images"], "| duplicates:", summary["duplicates"])

train_df = load_split("train")
fig, axes = plt.subplots(3, 8, figsize=(16, 6.5))
for r, c in enumerate(CLASSES):
    sub = train_df[train_df.label == c].sample(8, random_state=0)
    for ax, (_, row) in zip(axes[r], sub.iterrows()):
        ax.imshow(load_rgb(row.abs_path)); ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(row.source_class, fontsize=8)
    axes[r, 0].set_ylabel(c, fontsize=11)
fig.suptitle("Training samples per bin (title = original source class)"); plt.tight_layout(); plt.show()

## 2. The shared fine-tuning recipe (`finetune.py`)

* **Preprocessing** (what the ImageNet weights expect): centre square crop → 224×224 → normalise with the ImageNet mean/std.
* **Augmentation** (training only): random crop/zoom, horizontal and vertical flips, ±20° rotation, colour jitter, random blur.
* **Stage 1:** the backbone is frozen and only the new 3-class head is trained for 3 epochs (this also measures how good the
  frozen ImageNet features already are).
* **Stage 2:** the whole network is fine-tuned for up to 15 epochs, backbone learning rate = head learning rate / 10.
* **Loss and optimiser:** class-weighted cross-entropy with label smoothing 0.1, AdamW, cosine schedule, mixed precision on GPU.
* **Against overfitting:** augmentation, weight decay, label smoothing, the frozen first stage, and early stopping on validation
  macro-F1 (patience 5, best weights kept).

The output format is the same for all three: one probability per bin.

In [ ]:
def show_training(meta):
    ax_l, ax_m = plot_history(meta["history"], f"{meta['architecture']}: training vs validation")
    for ax in (ax_l, ax_m):
        ax.axvline(meta["stage1_epochs_run"] + 0.5, color="grey", linestyle=":", label="backbone unfrozen")
    ax_l.legend(); plt.show()
    print(f"{meta['params_millions']:.2f} M parameters | {meta['macs_millions']:.0f} M multiply-adds per image | "
          f"val macro-F1 head only {meta['stage1_best_val_f1']:.2f}% -> fine-tuned {meta['val']['macro_f1']:.2f}% | "
          f"best epoch {meta['best_epoch']} | {meta['train_time_min']:.1f} min on {meta['device']}")

## 3. Model A: MobileNetV3-Large (Howard et al., 2019)

Designed for phones: depthwise-separable convolutions (one 3×3 filter per channel, then a 1×1 convolution that mixes channels), squeeze-and-excitation (per-image channel re-weighting) and the h-swish activation; the block layout was found by neural architecture search. Head: the `classifier` block, its last layer replaced by 1280 → 3.

In [ ]:
import mobilenet_v3
meta_a = mobilenet_v3.train(head_epochs=1 if QUICK else 3, epochs=1 if QUICK else 15)
show_training(meta_a)

## 4. Model B: EfficientNet-B0 (Tan & Le, 2019)

MBConv blocks (inverted residual with a depthwise convolution, squeeze-and-excitation and swish) found by neural architecture search, in a family grown by compound scaling of depth, width and resolution; B0 is the smallest. Head: `classifier` (dropout + linear), the linear layer replaced by 1280 → 3.

In [ ]:
import efficientnet_b0
meta_b = efficientnet_b0.train(head_epochs=1 if QUICK else 3, epochs=1 if QUICK else 15)
show_training(meta_b)

## 5. Model C: ResNet-18 (He et al., 2016)

The classic residual network: pairs of full 3×3 convolutions, each added to its own input through a skip connection, which lets the gradient reach the early layers. The largest and oldest of the three, the baseline the mobile designs try to beat on speed. Head: `fc`, replaced by 512 → 3.

In [ ]:
import resnet18
meta_c = resnet18.train(head_epochs=1 if QUICK else 3, epochs=1 if QUICK else 15)
show_training(meta_c)

## 6. Experiments and results (same test split for all three)

1. Main comparison: top-1 accuracy, macro-F1, confusion matrix, per-class F1
2. Efficiency: runtime per image (CPU and GPU), parameters (M), multiply-adds (MACs), training time (min)
3. Per-source accuracy: web photos (Garbage Classification) vs real landfill photos (RealWaste)
4. Robustness: blur, darker, brighter, noise, 30° rotation (what a phone/webcam does to an image)
5. Frozen vs fine-tuned: validation macro-F1 with only the new head trained vs after fine-tuning the whole network
6. Our own local photos (`data/raw/local/<bin>/`), if provided

**MACs** (multiply-accumulate operations per image) measure compute; CPU speed also depends on how memory-bound the layers are,
so it does not always follow the MACs or the parameter count.

In [ ]:
import evaluate
out = evaluate.evaluate()
display(Markdown((RESULTS_DIR / "results_table.md").read_text()))

In [ ]:
res = out["results"]
display(pd.DataFrame({METHOD_INFO[k]["short"]: {c: v["per_class"][c]["f1"] for c in CLASSES} for k, v in
                      ((k, r["test"]) for k, r in res.items())}).round(1).rename_axis("per-class F1 (%)"))
display(pd.DataFrame({METHOD_INFO[k]["short"]: r["per_source_accuracy"] for k, r in res.items()}).round(1).rename_axis("accuracy (%) by source"))
if out["robustness_macro_f1"]:
    display(pd.DataFrame({METHOD_INFO[k]["short"]: v for k, v in out["robustness_macro_f1"].items()}).round(1).rename_axis("macro-F1 (%) under corruption"))
display(pd.DataFrame({METHOD_INFO[k]["short"]: {"head only (frozen ImageNet features)": r["val_macro_f1_head_only"],
                                                "fine-tuned": r["val_macro_f1"]} for k, r in res.items()}).round(2).rename_axis("validation macro-F1 (%)"))
if any("local_test" in r for r in res.values()):
    display(pd.DataFrame({METHOD_INFO[k]["short"]: {"accuracy": r["local_test"]["accuracy"], "macro F1": r["local_test"]["macro_f1"]}
                          for k, r in res.items() if "local_test" in r}).round(1).rename_axis("our local photos"))

In [ ]:
for name in ["confusion_matrices.png", "training_curves.png", "robustness.png"]:
    if (RESULTS_DIR / name).exists():
        display(ShowImage(filename=str(RESULTS_DIR / name)))

In [ ]:
evaluate.prediction_grid("test", n=12); plt.show()
for k in res:
    p = RESULTS_DIR / f"misclassified_{k}.png"
    if p.exists():
        display(ShowImage(filename=str(p)))

## 7. Recommendation used by the app

The app recommends the **fastest model on the CPU among those within 1 macro-F1 point of the most accurate one**: with about
1,000 test images a 1-point gap is about the size of the test-set noise, while speed decides how smooth the live camera feels.

In [ ]:
display(Markdown(out["recommendation"]["explanation"]))

## 8. The application

The app runs all three models on an uploaded image, an uploaded video, or the live webcam, draws the result on the input and
names the recommended model (read from `results/results.json`). In the live modes each model's answer is averaged over the last
few frames, so the label does not flicker.

```
python app.py      # web app (photo / live camera / video / compare / about)
python live.py     # full-window OpenCV webcam demo; keys 1/2/3 save the frame as a biodegradable/recyclable/residual photo
```